In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
from torch.optim import AdamW
from sklearn.metrics import accuracy_score
import torch.nn.functional as F

In [ ]:
class PromptedDataset(Dataset):
    def __init__(self, texts, labels, prompt="Classify: "):
        # Inject prompt to each input to simulate instruction-tuning behavior
        self.texts = [prompt + t for t in texts]
        self.labels = labels
        self.tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        # Tokenize the prompted input
        encoding = self.tokenizer(self.texts[idx], truncation=True, padding='max_length', max_length=128, return_tensors='pt')
        return {
            'input_ids': encoding['input_ids'].squeeze(),  # Tensor of input token IDs
            'attention_mask': encoding['attention_mask'].squeeze(),  # Tensor of attention mask
            'label': torch.tensor(self.labels[idx])  # Ground-truth label
        }

In [ ]:
class CustomBertClassifier(nn.Module):
    def __init__(self, adapter_size=64):
        super().__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')

        # Lightweight adapter layer with residual connection
        self.adapter = nn.Sequential(
            nn.Linear(768, adapter_size),
            nn.ReLU(),
            nn.Linear(adapter_size, 768)
        )

        # Final classifier for binary classification
        self.classifier = nn.Linear(768, 2)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0, :]  # Use [CLS] token as representation

        # Add adapter output via residual connection for task-specific transformation
        adapted = cls + self.adapter(cls)

        return self.classifier(adapted), adapted  # Return logits and feature representation


In [ ]:
def train_model(model, dataloader, optimizer, temperature=0.07, alpha=0.3):
    model.train()
    for batch in dataloader:
        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']
        labels = batch['label']

        logits, features = model(input_ids, attention_mask)

        # Step 1: Cross-entropy loss without reduction (loss per sample)
        loss_ce = F.cross_entropy(logits, labels, reduction='none')

        # Step 2: Compute weights based on relative difficulty (loss magnitude)
        weights = (loss_ce - loss_ce.min()) / (loss_ce.max() - loss_ce.min() + 1e-8)

        # Step 3: Reweight the cross-entropy loss
        loss_main = (loss_ce * weights.detach()).mean()

        # Step 4: Compute cosine similarity matrix for all pairs
        sim_matrix = F.cosine_similarity(features.unsqueeze(1), features.unsqueeze(0), dim=2)

        # Step 5: Create contrastive labels — 1 if same class, 0 otherwise
        labels_expanded = labels.unsqueeze(0) == labels.unsqueeze(1)

        # Step 6: Contrastive loss: encourage similar features for same-class samples
        contrastive_loss = F.cross_entropy(sim_matrix / temperature, labels_expanded.float())

        # Final loss combines main task + auxiliary contrastive signal
        loss = loss_main + alpha * contrastive_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


In [ ]:
from tqdm import tqdm

In [ ]:
texts = ["I love this!", "Terrible movie.", "Great plot", "Worst ever", "Fantastic!"]
labels = [1, 0, 1, 0, 1]  # 1 = Positive, 0 = Negative

dataset = PromptedDataset(texts, labels)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

model = CustomBertClassifier()
optimizer = AdamW(model.parameters(), lr=2e-5)

# Train for 5 epochs
for epoch in tqdm(range(5)):
    train_model(model, dataloader, optimizer)
    print(f"Epoch {epoch + 1} complete.")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

Epoch 3 complete.


 80%|████████  | 4/5 [00:25<00:05,  5.99s/it]

Epoch 4 complete.


100%|██████████| 5/5 [00:31<00:00,  6.25s/it]

Epoch 5 complete.


In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

def evaluate_model(model, dataloader):
    model.eval()  # Set model to eval mode (disables dropout, etc.)

    all_preds = []
    all_labels = []

    with torch.no_grad():  # No gradient calculation needed during eval
        for batch in dataloader:
            input_ids = batch['input_ids']
            attention_mask = batch['attention_mask']
            labels = batch['label']

            logits, _ = model(input_ids, attention_mask)
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Compute metrics
    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='macro')
    micro_precision, micro_recall, micro_f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='micro')
    cm = confusion_matrix(all_labels, all_preds)

    print("\n📊 Evaluation Metrics:")
    print(f"  Accuracy         : {acc:.4f}")
    print(f"  Macro Precision  : {precision:.4f}")
    print(f"  Macro Recall     : {recall:.4f}")
    print(f"  Macro F1 Score   : {f1:.4f}")
    print(f"  Micro F1 Score   : {micro_f1:.4f}")
    print(f"  Confusion Matrix :\n{cm}")

    return {
        "accuracy": acc,
        "macro_f1": f1,
        "micro_f1": micro_f1,
        "precision": precision,
        "recall": recall
    }


In [ ]:
test_texts = ["Amazing!", "I hated it.", "Not bad", "Best film", "Awful experience"]
test_labels = [1, 0, 1, 1, 0]

test_dataset = PromptedDataset(test_texts, test_labels)
test_dataloader = DataLoader(test_dataset, batch_size=2)

# Evaluate after training
metrics = evaluate_model(model, test_dataloader)



📊 Evaluation Metrics:
  Accuracy         : 0.4000
  Macro Precision  : 0.2000
  Macro Recall     : 0.5000
  Macro F1 Score   : 0.2857
  Micro F1 Score   : 0.4000
  Confusion Matrix :
[[2 0]
 [3 0]]


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
